## Dựa trên bài lab đã học ở tuần trước (CNN thuần) , hãy thay đỗi thành các tập dữ liệu khác như : 
- Cat and Dog 
- CIFAR-10 
- PlantVillage 
## Xử dụng các phương pháp phân tích dữ liệu , cân bằng dữ liệu,.. .
## Cố gắng tháy đỗi các tham số sao cho độ chỉnh xác lớn hơn 90% và tránh trường hợp overfitting. 
## KHông sử dụng các mô hình re-train hoặc các mô hình như Resnet,Convnext tiny,....
## Deadline:21/03/2026

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import os

In [14]:
print("Train Cat:", len(os.listdir("dataset/train/cat")))
print("Train Dog:", len(os.listdir("dataset/train/dog")))

print("Test Cat:", len(os.listdir("dataset/test/cat")))
print("Test Dog:", len(os.listdir("dataset/test/dog")))

Train Cat: 10000
Train Dog: 10000
Test Cat: 2500
Test Dog: 2500


In [15]:
transform_train = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
])

transform_test = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])

In [16]:
train_dataset = ImageFolder(
    "dataset/train",
    transform=transform_train
)

test_dataset = ImageFolder(
    "dataset/test",
    transform=transform_test
)

In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [19]:
print(train_dataset.classes)

['cat', 'dog']


In [20]:
class CNN(nn.Module):

    def __init__(self):

        super(CNN,self).__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(

            nn.Flatten(),
            nn.Linear(128*16*16,256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256,2)
        )

    def forward(self,x):

        x = self.conv(x)
        x = self.fc(x)

        return x

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [22]:
epochs = 20

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0

    for images,labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        _,pred = torch.max(outputs,1)

        total += labels.size(0)
        correct += (pred==labels).sum().item()

    acc = 100 * correct / total

    print("Epoch:",epoch,"Accuracy:",acc)

Epoch: 0 Accuracy: 61.48
Epoch: 1 Accuracy: 66.36
Epoch: 2 Accuracy: 68.1
Epoch: 3 Accuracy: 71.625
Epoch: 4 Accuracy: 74.035
Epoch: 5 Accuracy: 76.26
Epoch: 6 Accuracy: 77.805
Epoch: 7 Accuracy: 79.07
Epoch: 8 Accuracy: 80.56
Epoch: 9 Accuracy: 81.68
Epoch: 10 Accuracy: 82.72
Epoch: 11 Accuracy: 83.935
Epoch: 12 Accuracy: 84.475
Epoch: 13 Accuracy: 85.105
Epoch: 14 Accuracy: 85.615
Epoch: 15 Accuracy: 86.52
Epoch: 16 Accuracy: 86.515
Epoch: 17 Accuracy: 86.88
Epoch: 18 Accuracy: 87.945
Epoch: 19 Accuracy: 88.205


In [23]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images,labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _,pred = torch.max(outputs,1)

        total += labels.size(0)
        correct += (pred==labels).sum().item()

print("Test Accuracy:",100*correct/total)

Test Accuracy: 90.58
